# Malaika — Conversational IMCI Assessment on Colab

**Run on Colab with T4 GPU.**

This notebook launches the conversational Malaika app:
1. Installs dependencies
2. Loads Gemma 4 E4B via Unsloth (vision works!)
3. Launches Gradio chat interface
4. Gemma 4 analyzes photos and guides IMCI assessment

In [ ]:
# Cell 1: Install dependencies (kernel will restart at the end)
!rm -rf /content/malaika
!git clone https://github.com/Vimalk0703/Google-Gemma4-good-hackathon-VimalXMark /content/malaika
%cd /content/malaika
!pip install -q unsloth
!pip install -q --no-deps trl datasets librosa soundfile Pillow
!pip install -q gradio structlog pyyaml
import os; os._exit(0)

In [ ]:
# Cell 2: Auth + Load model + Launch app
# Run this after kernel restarts from Cell 1

%cd /content/malaika

from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # Replace with your token

# --- Load Gemma 4 via Unsloth (preserves vision encoder) ---
from unsloth import FastModel
import torch

print("Loading Gemma 4 E4B...")
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)
print(f"Model loaded | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("google/gemma-4-E4B-it")
print("Processor loaded")

# --- Quick vision test ---
from PIL import Image
import time

img = Image.new("RGB", (128, 128), color=(200, 150, 100))
messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "What color is this? One word."}]}]
input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=input_text, images=[img], return_tensors="pt").to(model.device)
t0 = time.time()
out = model.generate(**inputs, max_new_tokens=10, do_sample=False, repetition_penalty=1.3)
resp = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(f"Vision test: {time.time()-t0:.0f}s | {resp}")
print("Vision is working!" if resp.strip() else "WARNING: Vision not working")

In [ ]:
# Cell 3: Launch conversational chat app

import types, os
import malaika.chat_app as chat_module
from malaika.chat_app import ChatSession
from malaika.config import load_config
import gradio as gr

config = load_config()
session = ChatSession(config)

# Wire the Unsloth model into the session
session.model_loaded = True

def _analyze_image_direct(image_path, question):
    """Analyze image with Gemma 4 vision via Unsloth."""
    try:
        img = Image.open(image_path).convert("RGB")
        w, h = img.size
        if max(w, h) > 512:
            s = 512 / max(w, h)
            img = img.resize((int(w * s), int(h * s)))
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": question},
        ]}]
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(text=input_text, images=[img], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=False, repetition_penalty=1.3)
        return processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"Image analysis error: {e}")
        return ""

def _ask_gemma(question):
    """Text-only question to Gemma 4."""
    try:
        inputs = processor(text=question, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=150, do_sample=True, temperature=0.3, repetition_penalty=1.3)
        return processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    except Exception as e:
        print(f"Text generation error: {e}")
        return ""

session.analyze_image_direct = _analyze_image_direct
session.ask_gemma = _ask_gemma

# Load prompts
from malaika.prompts import (
    PromptRegistry, breathing, danger_signs, diarrhea,
    fever, heart, nutrition, speech, system, treatment,
)
print(f"Prompts: {len(PromptRegistry.list_all())}")

def respond(message, history):
    return chat_module.process_message(message, history, session)

chatbot = gr.Chatbot(height=500, type="messages", label="Malaika")
with gr.Blocks(title="Malaika", fill_height=True) as app:
    gr.HTML(
        '<div style="text-align:center;padding:12px;'
        'background:linear-gradient(135deg,#e8f4fd,#f0f7ff);'
        'border-bottom:3px solid #2979b9;border-radius:12px 12px 0 0;">'
        '<h1 style="margin:0;color:#1a5276;">Malaika</h1>'
        '<p style="color:#2e86c1;">WHO IMCI Child Health AI powered by Gemma 4</p>'
        '<p style="color:#7f8c8d;font-size:0.85em;">Not for clinical use</p></div>'
    )
    gr.ChatInterface(
        fn=respond, chatbot=chatbot, multimodal=True,
        textbox=gr.MultimodalTextbox(
            placeholder="Type your message or upload a photo...",
            file_count="single", file_types=["image"], sources=["upload"],
        ),
        title=None,
    )

print("\nLaunching Malaika...")
app.launch(share=True, server_name="0.0.0.0", server_port=7860, show_error=True)